# BM25 Lexical Search

**Definition:** BM25(Best Matching 25) is *lexical* search — the keyword-matching counterpart to the embedding search from the previous notebook. No model, no API calls, no vectors: it scores a document by how often the query's words appear in it, weighted by how rare those words are across the whole corpus. It's the classic algorithm behind search engines like Elasticsearch before vector search became common.

Three ideas do all the work:

- **Term Frequency (TF)**: Measures how often a word appears in a document. BM25 uses saturation so extra repeats add less value over time.
- **Inverse Document Frequency (IDF)**: Gives higher weight to rare words and lower weight to common words.
- **Document Length Normalization**: Prevents long documents from dominating results just because they contain more words.

**When to Use BM25**

- Finding exact words, product codes, or medical and legal terms.
- Fast text search without needing heavy GPUs or complex training.
- As the lexical component in hybrid search systems alongside vector embeddings.

**The trade-off vs. embeddings:** BM25 nails exact tokens an embedding blurs (product codes, error IDs, names like `CTX204b-P2A-001`), but is blind to synonyms — search "car" and it won't find "automobile". That's exactly why the next notebook combines both.

The class below is a full implementation, including input validation — the constructor and guard-clause lines can be skimmed; `_compute_bm25_score` and `search` are where the three ideas above actually happen.


### what this code does

- **`__init__`**: Sets up an empty index — nothing calculated yet.
- **`_default_tokenizer`**: Turns text into a list of lowercase words.
- **`add_document` / `add_documents`**: Feeds documents in and tallies which words appear where.
- **`_build_index` / `_calculate_idf`**: Figures out which words are rare (important) vs. common (unimportant).
- **`_compute_bm25_score`**: Scores one document by how well its words match the query.
- **`search`**: Scores every document, ranks them, and returns the best matches.

**One-line summary:** a smart word-counting system that finds documents matching your exact search words, favoring rare/meaningful words over common ones.

In [2]:
# BM25Index implementation
import math
import re
from collections import Counter
from typing import Callable, Optional, Any, List, Dict, Tuple


class BM25Index:
    def __init__(self, k1: float = 1.5, b: float = 0.75, tokenizer: Optional[Callable[[str], List[str]]] = None):
        self.documents: List[Dict[str, Any]] = []
        self._corpus_tokens: List[List[str]] = []
        self._doc_len: List[int] = []
        self._doc_freqs: Dict[str, int] = {}
        self._avg_doc_len: float = 0.0
        self._idf: Dict[str, float] = {}
        # Corpus stats go stale whenever a document is added, so the index is
        # rebuilt lazily on the next search rather than on every insert.
        self._index_built: bool = False

        self.k1 = k1
        self.b = b
        self._tokenizer = tokenizer if tokenizer else self._default_tokenizer

    def _default_tokenizer(self, text: str) -> List[str]:
        # Lowercase and split on anything non-alphanumeric. Deliberately naive:
        # no stemming and no stopword list, so "running" and "run" stay different.
        text = text.lower()
        tokens = re.split(r"\W+", text)
        return [token for token in tokens if token]

    def _update_stats_add(self, doc_tokens: List[str]):
        self._doc_len.append(len(doc_tokens))
        seen_in_doc = set()
        for token in doc_tokens:
            if token not in seen_in_doc:
                self._doc_freqs[token] = self._doc_freqs.get(token, 0) + 1
                seen_in_doc.add(token)
        self._index_built = False

    def _calculate_idf(self):
        # Rare terms get a big score, terms in nearly every document score
        # near zero. The +0.5 / +1 terms are standard BM25 smoothing.
        N = len(self.documents)
        self._idf = {}
        for term, freq in self._doc_freqs.items():
            self._idf[term] = math.log(((N - freq + 0.5) / (freq + 0.5)) + 1)

    def _build_index(self):
        if not self.documents:
            self._avg_doc_len = 0.0
            self._idf = {}
            self._index_built = True
            return
        self._avg_doc_len = sum(self._doc_len) / len(self.documents)
        self._calculate_idf()
        self._index_built = True

    def add_document(self, document: Dict[str, Any]):
        content = document["content"]
        doc_tokens = self._tokenizer(content)
        self.documents.append(document)
        self._corpus_tokens.append(doc_tokens)
        self._update_stats_add(doc_tokens)

    def add_documents(self, documents: List[Dict[str, Any]]):
        for document in documents:
            self.add_document(document)

    def _compute_bm25_score(self, query_tokens: List[str], doc_index: int) -> float:
        score = 0.0
        doc_term_counts = Counter(self._corpus_tokens[doc_index])
        doc_length = self._doc_len[doc_index]

        for token in query_tokens:
            if token not in self._idf:
                continue
            idf = self._idf[token]
            term_freq = doc_term_counts.get(token, 0)
            numerator = idf * term_freq * (self.k1 + 1)
            denominator = term_freq + self.k1 * (1 - self.b + self.b * (doc_length / self._avg_doc_len))
            score += numerator / (denominator + 1e-9)

        return score

    def search(self, query_text: str, k: int = 1, score_normalization_factor: float = 0.1) -> List[Tuple[Dict[str, Any], float]]:
        # Returns (document, score) pairs where LOWER is better. Raw BM25 is
        # higher-is-better and unbounded, so it's mapped through exp(-factor*raw)
        # into (0, 1] to match the cosine *distance* convention from VectorIndex.
        if not self.documents:
            return []

        if not self._index_built:
            self._build_index()
        if self._avg_doc_len == 0:
            return []

        query_tokens = self._tokenizer(query_text)
        if not query_tokens:
            return []

        raw_scores = []
        for i in range(len(self.documents)):
            raw_score = self._compute_bm25_score(query_tokens, i)
            if raw_score > 1e-9:
                raw_scores.append((raw_score, self.documents[i]))

        raw_scores.sort(key=lambda item: item[0], reverse=True)

        normalized_results = [
            (doc, math.exp(-score_normalization_factor * raw_score))
            for raw_score, doc in raw_scores[:k]
        ]
        normalized_results.sort(key=lambda item: item[1])

        return normalized_results

    def __len__(self) -> int:
        return len(self.documents)

    def __repr__(self) -> str:
        return f"BM25Index(count={len(self)}, k1={self.k1}, b={self.b}, index_built={self._index_built})"


In [3]:
with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)
print(f"{len(chunks)} chunks")


15 chunks


In [4]:
# No embedding function and no API key needed — the whole index is built
# from the text itself. Corpus statistics are computed lazily on first search.
store = BM25Index()
store.add_documents([{"content": chunk} for chunk in chunks])

store


BM25Index(count=15, k1=1.5, b=0.75, index_built=False)

## Searching by exact tokens

This query leans on exact tokens (`ctx`, `204b`, `biomarker`). They appear in essentially one section, so IDF gives them a large weight and that section wins easily — the case where BM25 beats embedding search outright.

Try re-running with a paraphrase that shares no words with the document, e.g. `"how did the new drug trial go?"`, to see the flip side: BM25 has no notion of meaning, so it scores poorly. Combining both retrievers is the subject of the next notebook.


In [5]:
query = "CTX-204b biomarker results"
results = store.search(query, k=2)

print(f"Query: {query}\n")
for rank, (doc, score) in enumerate(results, start=1):
    preview = doc["content"][:300].replace("\n", " ")
    print(f"--- Rank {rank} | score {score:.4f} (lower is better) ---")
    print(preview + "...\n")


Query: CTX-204b biomarker results

--- Rank 1 | score 0.3977 (lower is better) ---
Section 9: Pharmaceutical Development - Compound CTX-204b Phase IIa Update  Promising results emerged from the Phase IIa clinical trial (`Trial ID: CTX204b-P2A-001`) for Compound CTX-204b, our lead candidate targeting Receptor Pathway Gamma-7. Interim analysis of data from the initial patient cohort...

--- Rank 2 | score 0.6182 (lower is better) ---
Executive Summary  This report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the cross-pollination of ideas and methodologies, driving innovation and addressing complex challe...

